In [0]:
table_df = spark.table(
    "accelerator.metadata.table_configs"
)
table_df.display()

In [0]:

schema_data = []

for row in table_df.collect():

    table_name = row["table_name"]
    table_name = f"accelerator.gold.bronze_{table_name}"
    # print(table_name)
    print(f"Checking {table_name}")

    if not spark.catalog.tableExists(table_name):
        print(f"Table not found: {table_name}")
        continue
    df = spark.table(table_name)
    # print(df)
    # print(df.schema)
    # print(df.schema.fields)
    for field in df.schema.fields:

        schema_data.append(
            (
                str(table_name).replace('accelerator.gold.bronze_', ''),
                field.name,
                str(field.dataType)
            )
        )
print(f"Schema Data: {schema_data}")

In [0]:
# from pyspark.sql.functions import current_timestamp

# current_schema_df = spark.createDataFrame(
#         schema_data,
#         [
#             "table_name",
#             "column_name",
#             "data_type"
#         ]
#     ).withColumn(
#         "captured_date",
#         current_timestamp()
#     )

In [0]:
from pyspark.sql.functions import current_timestamp
# creating snapshot of this
if len(schema_data) > 0:
    current_schema_df = spark.createDataFrame(
        schema_data,
        [
            "table_name",
            "column_name",
            "data_type"
        ]
    ).withColumn(
        "captured_date",
        current_timestamp()
    )

    history_df = spark.table(
    "accelerator.metadata.schema_history"
    )
    from pyspark.sql.functions import max

    latest_date = history_df.select(
        max("captured_date")
    ).collect()[0][0]


    print(latest_date)

    previous_schema_df = history_df.filter(
        history_df.captured_date == latest_date
    )

    new_columns = (
            current_schema_df
            .select(
                "table_name",
                "column_name"
                # "data_type"
            )
            .subtract(
                previous_schema_df.select(
                    "table_name",
                    "column_name"
                    #   "data_type"
                )
            )
        )
    new_columns.display()
    # detecting any data type changes
    datatype_changes = (
            current_schema_df.alias("c")
            .join(
                previous_schema_df.alias("p"),
                [
                    current_schema_df.table_name
                    == previous_schema_df.table_name,

                    current_schema_df.column_name
                    == previous_schema_df.column_name
                ]
            )
            .filter(
                current_schema_df.data_type
                != previous_schema_df.data_type
            )
        )

    datatype_changes.display()

    from pyspark.sql.functions import lit,current_timestamp

    drift_df = (
        new_columns
        .withColumn(
            "drift_type",
            lit("NEW_COLUMN")
        )
        # .withColumn(
        #     "old_data_type",
        #     lit(None)
        # )
        # .withColumnRenamed(
        #     "data_type",
        #     "new_data_type"
        # )
        .withColumn(
            "detected_on",
            current_timestamp()
        )
    )
    drift_df.display()
    drift_df.select(
            "table_name",
            "drift_type",
            "column_name",
            # "old_data_type",
            # "new_data_type",
            "detected_on"
        ).write.mode("append").saveAsTable(
            "accelerator.metadata.schema_drift_audit"
        )
    from pyspark.sql.functions import lit,current_timestamp

    datatype_drift_df = (
        datatype_changes
        .select(
            current_schema_df.table_name.alias("table_name"),
            current_schema_df.column_name.alias("column_name"),
            previous_schema_df.data_type.alias("old_data_type"),
            current_schema_df.data_type.alias("new_data_type")
        )
        .withColumn(
            "drift_type",
            lit("DATATYPE_CHANGED")
        )
        .withColumn(
            "detected_on",
            current_timestamp()
        )
    )

    datatype_drift_df.select(
            "table_name",
            "drift_type",
            "column_name",
            "old_data_type",
            "new_data_type",
            "detected_on"
        ).write.mode("append").saveAsTable(
            "accelerator.metadata.schema_drift_audit"
        )

    current_schema_df.write \
    .mode("append") \
    .saveAsTable(
        "accelerator.metadata.schema_history"
    )
else :
    print("Empty schema data")

In [0]:
# current_schema_df.display()

In [0]:
# current_schema_df.write \
#     .mode("append") \
#     .saveAsTable(
#         "accelerator.metadata.schema_history"
#     )

In [0]:
# spark.read.table('accelerator.metadata.schema_history').display()

In [0]:
# history_df = spark.table(
#     "accelerator.metadata.schema_history"
# )
# from pyspark.sql.functions import max

# latest_date = history_df.select(
#     max("captured_date")
# ).collect()[0][0]


# print(latest_date)

# previous_schema_df = history_df.filter(
#     history_df.captured_date == latest_date
# )

# previous_schema_df.display()

In [0]:
# previous_schema_df.display()
# current_schema_df.display()


In [0]:
# new_columns = (
#     current_schema_df
#     .select(
#         "table_name",
#         "column_name"
#         # "data_type"
#     )
#     .subtract(
#         previous_schema_df.select(
#             "table_name",
#             "column_name"
#             #   "data_type"
#         )
#     )
# )
# new_columns.display()

In [0]:
# # detecting any data type changes
# datatype_changes = (
#     current_schema_df.alias("c")
#     .join(
#         previous_schema_df.alias("p"),
#         [
#             current_schema_df.table_name
#             == previous_schema_df.table_name,

#             current_schema_df.column_name
#             == previous_schema_df.column_name
#         ]
#     )
#     .filter(
#         current_schema_df.data_type
#         != previous_schema_df.data_type
#     )
# )

# datatype_changes.display()

In [0]:
# display(new_columns)
# display(
#     datatype_changes.select(
#         current_schema_df.table_name.alias("Current_table_name"),
#         current_schema_df.column_name.alias("Current_column_name"),
#         previous_schema_df.data_type.alias("Previous_data_type"),
#         current_schema_df.data_type.alias("Current_data_type")
#     )
# )

In [0]:
# from pyspark.sql.functions import lit,current_timestamp

# drift_df = (
#     new_columns
#     .withColumn(
#         "drift_type",
#         lit("NEW_COLUMN")
#     )
#     # .withColumn(
#     #     "old_data_type",
#     #     lit(None)
#     # )
#     # .withColumnRenamed(
#     #     "data_type",
#     #     "new_data_type"
#     # )
#     .withColumn(
#         "detected_on",
#         current_timestamp()
#     )
# )
# drift_df.display()

In [0]:
# table_name	string	null
# drift_type	string	null
# column_name	string	null
# old_data_type	string	null
# new_data_type	string	null
# detected_on	timestamp	null
# drift_df.select(
#     "table_name",
#     "drift_type",
#     "column_name",
#     # "old_data_type",
#     # "new_data_type",
#     "detected_on"
# ).write.mode("append").saveAsTable(
#     "accelerator.metadata.schema_drift_audit"
# )

In [0]:
# from pyspark.sql.functions import lit,current_timestamp

# datatype_drift_df = (
#     datatype_changes
#     .select(
#         current_schema_df.table_name.alias("table_name"),
#         current_schema_df.column_name.alias("column_name"),
#         previous_schema_df.data_type.alias("old_data_type"),
#         current_schema_df.data_type.alias("new_data_type")
#     )
#     .withColumn(
#         "drift_type",
#         lit("DATATYPE_CHANGED")
#     )
#     .withColumn(
#         "detected_on",
#         current_timestamp()
#     )
# )

In [0]:
# datatype_drift_df.display()

In [0]:
# datatype_drift_df.select(
#     "table_name",
#     "drift_type",
#     "column_name",
#     "old_data_type",
#     "new_data_type",
#     "detected_on"
# ).write.mode("append").saveAsTable(
#     "accelerator.metadata.schema_drift_audit"
# )

In [0]:
# current_schema_df.write \
#     .mode("append") \
#     .saveAsTable(
#         "accelerator.metadata.schema_history"
#     )